<a href="https://colab.research.google.com/github/devNull-bootloader/Jugend-trainiert-Mathematik/blob/main/Aufgabe_1_04_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diese sind meine Lösungen für die Aufgaben über Logik, Beweise und Problemlösen.

---

## Aufgabe 1:

**Gegeben:**


---


- 17 Kinder mit nichtnegativen ganzen Zahlen $x_1,\dots,x_{17}$.
- Jede beliebige 5‑Gruppe hat höchstens 25 Eier.
- Jede beliebige 3‑Gruppe hat mindestens 14 Eier.

**Gesucht:**


---


- Die mögliche gerade Gesamtzahl $S=\sum_{i=1}^{17} x_i$.

**Strategie:**

---

1. Mathematische Schranken für $S$ per Doppelzählung ableiten.
2. Nur die verbleibenden geraden Kandidaten für $S$ algorithmisch prüfen.
3. Für jeden Kandidaten ein ILP formulieren, das prüft, ob eine Verteilung $x_1,\dots,x_{17}$ existiert.

### Erklärung der Doppelzählung:

Summe über alle 5‑Gruppen: Jede 5‑Gruppe hat ≤ 25 Eier, es gibt $\binom{17}{5}$ solche Gruppen. Also gilt:

$
\sum_{\text{alle 5‑Gruppen}} \text{Eier} \le \binom{17}{5}\cdot 25.
$

---


Jedes Kind erscheint in genau $\binom{16}{4}$ 5‑Gruppen, daher ist die linke Seite gleich $\binom{16}{4}\,S$. Daraus folgt:


$
\binom{16}{4}\,S \le \binom{17}{5}\cdot 25.
$


---

Summe über alle 3‑Gruppen: Jede 3‑Gruppe hat ≥ 14 Eier, es gibt $\binom{17}{3}$ solche Gruppen. Also gilt:


$
\binom{16}{2}\,S \ge \binom{17}{3}\cdot 14.
$

---

Diese beiden Ungleichungen liefern numerische Schranken für $S$. Danach runden wir auf ganze, gerade Werte und prüfen nur diese Kandidaten weiter.

In [ ]:
import math
from itertools import combinations
import numpy as np
import pulp

def C(n, k):
  return math.comb(n, k)

upper = (C(17,5)*25) / C(16,4)
lower = (C(17,3)*14) / C(16,2)
print(upper, "|", lower)

cand = [s for s in range(math.ceil(lower), math.floor(upper)+1) if s%2==0]
print("Kandidaten S:", cand)

85.0 | 79.33333333333333
Kandidaten S: [80, 82, 84]


**Erläuterung:**

---

Diese Zelle rechnet die beiden Ungleichungen aus und erzeugt die Menge der ganzzahligen, geraden Kandidaten für $S$. Wir verwenden diese Kandidaten als Input für die algorithmische Prüfung. Das reduziert den Suchraum drastisch.

In [ ]:
idx = range(17)
triples = list(combinations(idx,3))
quints  = list(combinations(idx,5))

def check_all_3(x):
    arr = np.array(x)
    return all(arr[list(t)].sum() >= 14 for t in triples)

def check_all_5(x):
    arr = np.array(x)
    return all(arr[list(q)].sum() <= 25 for q in quints)

**Erläuterung:**  

---

- Wir erzeugen `triples` und `quints` einmal, um wiederholte Kombinationserzeugung zu vermeiden.  
- `check_all_3` und `check_all_5` sind einfache, deterministische Prüfungen, die eine gegebene Verteilung $x$ gegen die Gruppenbedingungen testen.  
- Diese Funktionen sind nützlich zur Validierung von Lösungen, die der ILP‑Solver liefert, und können auch in heuristischen Suchen verwendet werden.


In [ ]:
def exists_distribution_for_S(S, time_limit=10):
    prob = pulp.LpProblem('eggs', pulp.LpStatusOptimal)
    x = [pulp.LpVariable(f"x{i}", lowBound=0, cat='Integer') for i in range(17)]
    # triple constraints
    for t in triples:
        prob += sum(x[i] for i in t) >= 14
    # quint constraints
    for q in quints:
        prob += sum(x[i] for i in q) <= 25
    prob += sum(x) == S
    prob.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit))
    return pulp.LpStatus[prob.status], [v.value() for v in x] if pulp.LpStatus[prob.status]=='Optimal' else None

for S in cand:
    status, sol = exists_distribution_for_S(S)
    print(S, status)
    if sol:
        print(sol)
        break

80 Infeasible
82 Infeasible
84 Optimal
[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 4.0, 5.0, 5.0, 5.0, 5.0]


**Erläuterung ILP:**  

---

- **Variablen:** $x_0,\dots,x_{16}$ ganzzahlig, ≥0.  
- **Nebenbedingungen:** Für jedes Tripel $\sum_{i\in T} x_i \ge 14$. Für jedes Quintett $\sum_{i\in Q} x_i \le 25$. Und die Summengleichung $\sum_i x_i = S$.  
- **Solver:** `pulp` mit CBC. `timeLimit` schützt vor zu langen Läufen.  
- **Warum ILP:** Die Nebenbedingungen sind linear und ganzzahlig; ILP ist eine saubere, deterministische Methode, um Existenz einer Verteilung zu prüfen, ohne alle Permutationen zu enumerieren.

### Lösung:

---

**Behauptung:** Es wurden **84** Ostereier bemalt.

**Beweis (schrittweise):**

1. **Notation.** Sei $x_1,\dots,x_{17}$ die von den 17 Kindern bemalten Eier und $S=\sum_{i=1}^{17}x_i$ die Gesamtzahl.

2. **Doppelzählung für 5‑Gruppen.** Die Summe der Eier über alle $\binom{17}{5}$ 5‑Kinder‑Gruppen ist höchstens $\binom{17}{5}\cdot 25$. Andererseits wird jedes Kind in genau $\binom{16}{4}$ solcher Gruppen gezählt, also gilt
$
\binom{16}{4}\,S \le \binom{17}{5}\cdot 25.
$

3. **Doppelzählung für 3‑Gruppen.** Die Summe der Eier über alle $\binom{17}{3}$ 3‑Kinder‑Gruppen ist mindestens $\binom{17}{3}\cdot 14$. Da jedes Kind in $\binom{16}{2}$ 3‑Gruppen vorkommt, folgt
$
\binom{16}{2}\,S \ge \binom{17}{3}\cdot 14.
$

4. **Einsetzen der Binomialkoeffizienten und numerische Schranken.** Mit $\binom{17}{5}=6188,\ \binom{16}{4}=1820,\ \binom{17}{3}=680,\ \binom{16}{2}=120$ erhält man
$
1820\,S \le 6188\cdot 25 =154700 \quad\Rightarrow\quad S\le 85,
$
$
120\,S \ge 680\cdot 14 =9520 \quad\Rightarrow\quad S\ge \frac{9520}{120}=79\frac{1}{3}.
$
Da $S$ ganzzahlig und zusätzlich gerade sein muss, bleiben nur die Kandidaten
$
S\in\{80,82,84\}.
$

5. **Ausschluss von 80 und 82.** Betrachte die kleinstmöglichen sinnvollen Einzelwerte: drei Kinder mit je 4 Eiern würden eine 3‑Gruppe mit Summe $12<14$ erzeugen. Um die 3‑Gruppenbedingung zu erfüllen, dürfen also nicht drei Kinder gleichzeitig den Wert 4 haben. Schreibe $S$ als Kombination von 4 und 5 (kleinste sinnvolle Werte): $S=4\cdot(17-k)+5k=68+k$, wobei $k$ die Anzahl der Kinder mit 5 Eiern ist.  
- Für $S=80$ wäre $k=12$, also gäbe es 5 Kinder mit 4 Eiern — daraus folgt zwangsläufig eine Dreiergruppe aus drei Kindern mit 4 Eiern, Widerspruch.  
- Für $S=82$ wäre $k=14$, also gäbe es 3 Kinder mit 4 Eiern — wiederum existiert eine Dreiergruppe mit Summe $12<14$, Widerspruch.

6. **Existenz für $S=84$.** Setze 16 Kinder mit je 5 Eiern und 1 Kind mit 4 Eiern. Dann ist
$
S=16\cdot 5 + 1\cdot 4 =84.
$
Jede 5‑Gruppe enthält höchstens fünf Kinder mit 5 Eiern, also hat jede 5‑Gruppe Summe $\le 25$. Jede 3‑Gruppe enthält mindestens zwei Kinder mit 5 und höchstens eine mit 4, also hat jede 3‑Gruppe Summe $\ge 5+5+4=14$. Damit sind beide Bedingungen erfüllt.

**Schlussfolgerung:** Die einzige mögliche gerade Gesamtzahl, die alle Bedingungen erfüllt, ist **$S=84$**.